# 🚌 MAGI — XGBoost Model: Data Explainer
**Multi-Answer Geographical Informations (MAGI)**  
One of three ensemble models: `Probit (Chen et al.)` · `XGBoost` · `Jayden Method`

This notebook walks through how data flows into the XGBoost model — what features are used, where they come from, and why.

---
## 1. Overview: What does the XGBoost model do?

The XGBoost model predicts **delay severity** for a TTC vehicle into one of 4 classes:

| Class | Label    | Meaning                     |
|-------|----------|-----------------------------|
| 0     | on_time  | No significant delay        |
| 1     | minor    | Small delay (1–5 min)       |
| 2     | moderate | Noticeable delay (5–15 min) |
| 3     | severe   | Major delay (15+ min)       |

It outputs **probabilities** for each class, not just a hard label — which allows MAGI to compare confidence across all three models using **log loss**.

Separate models are trained per transit **mode**: `bus`, `streetcar`, `subway`.

---
## 2. Feature Tiers

Features are organized into three tiers based on their source literature.

In [ ]:
import pandas as pd

feature_table = pd.DataFrame([
    # Tier 1 — Chen et al. 2007 (proven variables)
    {"Feature": "cumulative_dwell_time", "Tier": "Tier 1 — Chen et al.", "Chen β": 0.237, "Description": "Total time spent at stops so far (minutes). Strongest predictor of delay."},
    {"Feature": "cumulative_leg_time",   "Tier": "Tier 1 — Chen et al.", "Chen β": 0.186, "Description": "Total scheduled travel time elapsed (from shape_dist_traveled)."},
    {"Feature": "cumulative_stops",      "Tier": "Tier 1 — Chen et al.", "Chen β": 0.099, "Description": "Number of stops served so far in this trip."},
    {"Feature": "day_of_week",           "Tier": "Tier 1 — Chen et al.", "Chen β": None,  "Description": "0=Mon … 6=Sun. Weekday patterns differ significantly."},
    {"Feature": "section_id",            "Tier": "Tier 1 — Chen et al.", "Chen β": None,  "Description": "Route segment identifier (hash of route_id % 100)."},
    {"Feature": "rain_mm",               "Tier": "Tier 1 — Chen et al.", "Chen β": None,  "Description": "Rainfall in mm. Weather-sourced."},
    {"Feature": "wind_speed",            "Tier": "Tier 1 — Chen et al.", "Chen β": None,  "Description": "Wind speed km/h. Weather-sourced."},
    {"Feature": "visibility_km",         "Tier": "Tier 1 — Chen et al.", "Chen β": None,  "Description": "Visibility in km. Low visibility → delays."},
    # Tier 2 — Trépanier et al.
    {"Feature": "commercial_speed",      "Tier": "Tier 2 — Trépanier",   "Chen β": None,  "Description": "Vehicle's average operating speed on the route."},
    {"Feature": "occupancy_rate",        "Tier": "Tier 2 — Trépanier",   "Chen β": None,  "Description": "How full the vehicle is (0–1). Higher occupancy → longer dwell."},
    {"Feature": "schedule_adherence",    "Tier": "Tier 2 — Trépanier",   "Chen β": None,  "Description": "How on-time the vehicle has been across this trip (seconds)."},
    {"Feature": "hour_of_day",           "Tier": "Tier 2 — Trépanier",   "Chen β": None,  "Description": "Hour 0–23. Captures AM/PM peak patterns."},
    {"Feature": "is_peak_hour",          "Tier": "Tier 2 — Trépanier",   "Chen β": None,  "Description": "1 if 7–9am or 4–6pm, else 0."},
    # Tier 3 — Toronto specific
    {"Feature": "temperature_c",         "Tier": "Tier 3 — Toronto",      "Chen β": None,  "Description": "Temperature in Celsius. Extreme cold affects TTC buses."},
    {"Feature": "snow_mm",               "Tier": "Tier 3 — Toronto",      "Chen β": None,  "Description": "Snowfall mm. High impact in Toronto winters."},
    {"Feature": "is_holiday",            "Tier": "Tier 3 — Toronto",      "Chen β": None,  "Description": "1 on Ontario statutory holidays."},
    {"Feature": "is_special_event",      "Tier": "Tier 3 — Toronto",      "Chen β": None,  "Description": "1 if a major event is happening (e.g. Raptors game, CNE)."},
    {"Feature": "delay_trend",           "Tier": "Tier 3 — Toronto",      "Chen β": None,  "Description": "Rolling average delay for this route over the past hour."},
    {"Feature": "route_historical_avg",  "Tier": "Tier 3 — Toronto",      "Chen β": None,  "Description": "Historical average delay for this route at this time of day."},
])

feature_table.set_index("Feature", inplace=True)
feature_table.style.set_table_styles([
    {"selector": "th", "props": [("background", "#2c2c2c"), ("color", "white")]}
]).background_gradient(subset=["Chen β"], cmap="YlOrRd", na_rep="—")

---
## 3. Data Sources: Real vs Simulated

Training data comes from **two sources** that are combined before model training:

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Training Data: Real vs Simulated", fontsize=14, fontweight="bold")

# Left: sources
ax = axes[0]
sources = ["Real (GTFS\nRealtime)", "Simulated\n(GTFS Static)"]
colors  = ["#e74c3c", "#3498db"]
bars = ax.bar(sources, [1, 10], color=colors, width=0.5, edgecolor="white", linewidth=1.5)
ax.set_ylabel("Relative Volume (approx.)", fontsize=11)
ax.set_title("Volume Ratio")
ax.set_ylim(0, 12)
for bar, label in zip(bars, ["Live vehicle positions\nmatched to GTFS schedules",
                               "All stop_times rows\nwith simulated severity labels"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            label, ha="center", fontsize=9, color="#333")
ax.spines[["top","right"]].set_visible(False)

# Right: simulation logic
ax2 = axes[1]
ax2.axis("off")
logic = [
    ("Condition",                               "Score Δ",  "#555"),
    ("AM Peak  (7–9am)",                        "+2",       "#e67e22"),
    ("PM Peak  (4–6pm)",                        "+1",       "#e67e22"),
    ("cumulative_stops > 10",                   "+1",       "#8e44ad"),
    ("cumulative_stops > 20",                   "+1",       "#8e44ad"),
    ("cumulative_dwell_time > 8 min",           "+2",       "#c0392b"),
    ("is_sunday == 1",                          "−2",       "#27ae60"),
    ("Random noise",                            "N(0, 0.5)","#7f8c8d"),
    ("",                                        "",         "white"),
    ("Score ≤ 0  → class 0  (on_time)",        "",         "#2ecc71"),
    ("Score 1–2  → class 1  (minor)",          "",         "#f1c40f"),
    ("Score 3–4  → class 2  (moderate)",       "",         "#e67e22"),
    ("Score ≥ 5  → class 3  (severe)",         "",         "#e74c3c"),
]
ax2.set_title("Simulation Severity Logic", fontsize=11)
for i, (cond, delta, color) in enumerate(logic):
    ax2.text(0.02, 1 - i*0.075, cond,  fontsize=9,  color=color, transform=ax2.transAxes)
    ax2.text(0.75, 1 - i*0.075, delta, fontsize=9,  color=color, transform=ax2.transAxes, fontweight="bold")

plt.tight_layout()
plt.show()

---
## 4. Real Data Pipeline: GTFS Realtime → Training Record

This walks through exactly what happens for each live vehicle in `train_xgboost.py`:

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as FancyBboxPatch

fig, ax = plt.subplots(figsize=(12, 5))
ax.axis("off")

steps = [
    ("Live Vehicle\n(lat, lon,\nrouteTag)",       "#2980b9"),
    ("Bounding box\nnearby stops\n±0.01°",        "#8e44ad"),
    ("Haversine\ndist < 0.1 km?",                 "#16a085"),
    ("Match GTFS\nstop_times\nby route + stop",   "#d35400"),
    ("Find closest\nscheduled\narrival",           "#c0392b"),
    ("delay_seconds\n= now − scheduled",          "#7f8c8d"),
    ("classify_delay()\n→ severity 0–3",          "#27ae60"),
]

for i, (label, color) in enumerate(steps):
    x = i * 1.55
    box = plt.Rectangle((x, 0.3), 1.3, 0.9, color=color, alpha=0.85, zorder=2)
    ax.add_patch(box)
    ax.text(x + 0.65, 0.75, label, ha="center", va="center",
            fontsize=8.5, color="white", fontweight="bold", zorder=3)
    if i < len(steps) - 1:
        ax.annotate("", xy=(x + 1.35, 0.75), xytext=(x + 1.3, 0.75),
                    arrowprops=dict(arrowstyle="->", color="#555", lw=1.8))

ax.set_xlim(-0.1, len(steps) * 1.55)
ax.set_ylim(0, 1.5)
ax.set_title("Real Record Pipeline: Live Vehicle → Training Label", fontsize=13, fontweight="bold", pad=10)
plt.tight_layout()
plt.show()

---
## 5. Feature Engineering Details

Let's look at how the three **Chen et al. core features** are actually computed from GTFS data:

In [ ]:
import numpy as np
import pandas as pd

# Simulate a sample trip to show feature evolution
np.random.seed(42)
n_stops = 25

stop_seq         = np.arange(1, n_stops + 1)
dwell_per_stop   = np.random.uniform(0.2, 0.5, n_stops)   # minutes at each stop
dist_per_segment = np.random.uniform(0.3, 0.8, n_stops)   # km per segment

# As coded in train_xgboost.py:
# cumulative_dwell_time = stop_sequence * 0.3  (simplified proxy)
# cumulative_leg_time   = shape_dist_traveled  (cumulative km)
# cumulative_stops      = stop_sequence

cum_dwell = stop_seq * 0.3
cum_leg   = np.cumsum(dist_per_segment)
cum_stops = stop_seq

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Chen et al. Core Features — Evolution Across a Single Trip", fontsize=13, fontweight="bold")

for ax, y, label, color, beta in zip(
    axes,
    [cum_dwell, cum_leg, cum_stops],
    ["cumulative_dwell_time (min)", "cumulative_leg_time (km)", "cumulative_stops"],
    ["#e74c3c", "#3498db", "#2ecc71"],
    ["β=0.237", "β=0.186", "β=0.099"]
):
    ax.plot(stop_seq, y, color=color, lw=2.5)
    ax.fill_between(stop_seq, y, alpha=0.15, color=color)
    ax.set_xlabel("Stop Sequence", fontsize=10)
    ax.set_title(f"{label}\n{beta} (Chen et al.)", fontsize=10)
    ax.spines[["top","right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("These three features accumulate across the trip — the further the bus has gone, the more dwell time has built up, which is the primary driver of delay in Chen et al.'s model.")

---
## 6. Class Distribution: What does a balanced dataset look like?

Understanding the label distribution is critical — XGBoost can struggle with imbalanced classes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Reproduce simulate_delay() logic on a synthetic dataset
np.random.seed(42)
N = 10_000

hour          = np.random.randint(0, 24, N)
cum_stops     = np.random.randint(1, 40,  N)
cum_dwell     = cum_stops * 0.3
is_sunday     = (np.random.rand(N) < 0.14).astype(int)

score = np.zeros(N, dtype=float)
score += 2 * ((hour >= 7)  & (hour <= 9))
score += 1 * ((hour >= 16) & (hour <= 18))
score += 1 * (cum_stops > 10)
score += 1 * (cum_stops > 20)
score += 2 * (cum_dwell > 8)
score -= 2 * is_sunday
score += np.random.normal(0, 0.5, N)

labels = np.where(score <= 0, 0,
         np.where(score <= 2, 1,
         np.where(score <= 4, 2, 3)))

counts = [(labels == i).sum() for i in range(4)]
names  = ["0: on_time", "1: minor", "2: moderate", "3: severe"]
colors = ["#2ecc71", "#f1c40f", "#e67e22", "#e74c3c"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Simulated Delay Severity Distribution (N=10,000)", fontsize=13, fontweight="bold")

axes[0].bar(names, counts, color=colors, edgecolor="white", linewidth=1.3)
axes[0].set_ylabel("Count")
axes[0].set_title("Absolute Counts")
axes[0].spines[["top","right"]].set_visible(False)
for i, c in enumerate(counts):
    axes[0].text(i, c + 50, f"{c:,}", ha="center", fontsize=9)

axes[1].pie(counts, labels=names, colors=colors, autopct="%1.1f%%",
            startangle=140, wedgeprops={"edgecolor":"white","linewidth":1.5})
axes[1].set_title("Class Balance")

plt.tight_layout()
plt.show()

print("⚠️  If class 0 dominates, XGBoost may over-predict 'on_time'.")
print("    Consider: scale_pos_weight, class_weight, or stratified sampling.")

---
## 7. Mode Split: Bus vs Streetcar vs Subway

MAGI trains **separate** XGBoost models per transit mode. Here's the GTFS `route_type` mapping:

In [ ]:
mode_table = pd.DataFrame([
    {"route_type": 0, "Mode": "streetcar", "Toronto Example": "504 King, 501 Queen, 509 Harbourfront", "Why separate model?": "Fixed rail, boarding islands, different dwell patterns"},
    {"route_type": 1, "Mode": "subway",    "Toronto Example": "Line 1 (YUS), Line 2 (BD), Line 4 (Sheppard)", "Why separate model?": "Station dwell, headway-based, no traffic congestion"},
    {"route_type": 3, "Mode": "bus",       "Toronto Example": "All other routes (default)", "Why separate model?": "Largest mode, most weather/traffic sensitivity"},
])
mode_table.set_index("route_type")

---
## 8. Model Architecture

The hyperparameters used in `train_xgboost.py`:

In [ ]:
params = pd.DataFrame([
    {"Parameter": "n_estimators",     "Value": 300,              "Purpose": "Number of boosting rounds. Higher = more complex but slower."},
    {"Parameter": "max_depth",        "Value": 6,                "Purpose": "Max tree depth. Prevents overfitting."},
    {"Parameter": "learning_rate",    "Value": 0.05,             "Purpose": "Shrinkage. Small = more stable, needs more trees."},
    {"Parameter": "subsample",        "Value": 0.8,              "Purpose": "Row sampling per tree. Reduces variance."},
    {"Parameter": "colsample_bytree", "Value": 0.8,              "Purpose": "Feature sampling per tree. Prevents co-adaptation."},
    {"Parameter": "objective",        "Value": "multi:softprob", "Purpose": "Outputs class probabilities — required for log loss in MAGI."},
    {"Parameter": "num_class",        "Value": 4,                "Purpose": "on_time / minor / moderate / severe."},
    {"Parameter": "eval_metric",      "Value": "mlogloss",       "Purpose": "Monitored on eval set — same metric MAGI uses to pick winner."},
])
params.set_index("Parameter")

---
## 9. How MAGI Uses the XGBoost Output

After prediction, MAGI compares all three models using **log loss** and picks the winner:

In [ ]:
import math

# Example: actual severity = 2 (moderate)
actual = 2

model_outputs = {
    "probit":  {"on_time": 0.20, "minor": 0.35, "moderate": 0.30, "severe": 0.15},
    "xgboost": {"on_time": 0.05, "minor": 0.15, "moderate": 0.65, "severe": 0.15},
    "jayden":  {"on_time": 0.10, "minor": 0.10, "moderate": 0.70, "severe": 0.10},
}

print(f"Actual severity: {actual} (moderate)\n")
print(f"{'Model':<10} {'P(moderate)':<14} {'Log Loss':<10} {'Winner?'}")
print("-" * 48)

best_model, best_ll = None, float("inf")
for name, probs in model_outputs.items():
    prob_list = list(probs.values())
    ll = -math.log(prob_list[actual] + 1e-10)
    is_best = ll < best_ll
    if is_best:
        best_model, best_ll = name, ll

for name, probs in model_outputs.items():
    prob_list = list(probs.values())
    ll = round(-math.log(prob_list[actual] + 1e-10), 4)
    winner = "✅ WINNER" if name == best_model else ""
    print(f"{name:<10} {prob_list[actual]:<14.2f} {ll:<10} {winner}")

print(f"\nMAGI selects: {best_model} (lowest log loss = highest confidence on correct class)")

---
## 10. Quick Reference: `xgboost_predict()` Input/Output

Summary of how to call the model in production:

In [ ]:
# Example features dict — what MAGI passes in
example_features = {
    # Tier 1 (Chen et al.) — most impactful
    "cumulative_dwell_time": 4.5,   # 15 stops * 0.3 min
    "cumulative_leg_time":   8.2,   # shape_dist_traveled km
    "cumulative_stops":      15,
    "day_of_week":           1,     # Tuesday
    "section_id":            42,
    "rain_mm":               0,
    "wind_speed":            12,
    "visibility_km":         10,
    # Tier 2
    "commercial_speed":      18,
    "occupancy_rate":        0.6,
    "schedule_adherence":    -90,   # 90 seconds late
    "hour_of_day":           8,     # AM peak
    "is_peak_hour":          1,
    # Tier 3
    "temperature_c":         -5,
    "snow_mm":               2,
    "is_holiday":            0,
    "is_special_event":      0,
    "delay_trend":           45,    # route trending 45s late
    "route_historical_avg":  30,
    # Mode
    "mode":                  "bus"
}

print("📥  Input to xgboost_predict(features, mode='bus'):")
for k, v in example_features.items():
    print(f"    {k:<28} = {v}")

print("\n📤  Expected output shape:")
print("    {")
print("      'model':     'xgboost',")
print("      'predicted': 2,          # int 0-3")
print("      'probabilities': {")
print("          'on_time':  0.05,")
print("          'minor':    0.15,")
print("          'moderate': 0.65,    # ← predicted class")
print("          'severe':   0.15")
print("      }")
print("    }")

---
## Summary

| Step | What happens |
|------|--------------|
| **Data** | Real GTFS Realtime + simulated GTFS Static, combined 80/20 split |
| **Features** | 19 features across 3 tiers: Chen et al., Trépanier, Toronto-specific |
| **Training** | Separate `XGBClassifier` per mode (bus / streetcar / subway) |
| **Output** | 4-class probability vector: on_time / minor / moderate / severe |
| **MAGI** | Picks winning model by lowest log loss on the actual severity |

The XGBoost model is the most feature-rich of the three, but the Probit (Chen et al.) baseline and Jayden Method can outperform it when training data is sparse or when the pattern is simple (e.g. very long dwell time → obviously severe).